In [1]:
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from helper_functions.data_pipeline import DataPipeline
from utils.dataset_processor import DatasetProcessor

Error: 

In [ ]:
# Dataset list (animalID, date, server)
datasets = [
    ("HA1-00", "2023-06-27", 'V:'),
    ("HA1-00", "2023-06-29", 'V:'),
    ("HA1-00", "2023-07-07", 'W:'),  # squinty after 30k
    ("HA10-1L", "2023-03-31", 'V:'),
    ("HA10-1L", "2023-04-10", 'V:'),  # used in Mabry's thesis
    ("HA10-1L", "2023-04-12", 'V:'),
    ("HA10-1L", "2023-04-17", 'V:'),  # difficult dataset reflections/squinty
    ("HA11-1R", "2023-04-07", 'V:'),  # inconsistent brightness?
    ("HA11-1R", "2023-04-13", 'V:'),
    ("HA11-1R", "2023-05-01", 'V:'),
    ("HA11-1R", "2023-05-02", 'V:'),
    ("HA11-1R", "2023-05-05", 'V:'),
    ("HA2-1L", "2023-04-12", 'V:'),  # squinty
    # ("HA2-1L", "2023-04-28", 'V:'),  # occlusion; likely missing frames
    ("HA2-1L", "2023-05-01", 'V:'),  # unreliable after 37k
    ("HA2-1L", "2023-05-05", 'V:'),
    # ("HE1-00", "2023-05-30", 'V:'),
    # ("HA1-00", "2023-08-28", 'W:'),
    # ("HA1-00", "2023-08-25", 'W:'),
]

pipeline = DataPipeline()

# Load the data from the datasets list
data_loaders, celltype_info = pipeline.load_data(datasets, load_celltypes=True)

print(f"Loaded {len(data_loaders)} datasets")

In [ ]:
# Align neural data around task events
processor = DatasetProcessor(alignment={
    'type': 'pre',
    'data_type': 'dff'
})

aligned_data = processor.process_datasets(
    data_loaders=data_loaders,
    celltype_info=celltype_info,
    save_path='aligned_data.pkl'
)

for (animalID, date), data in aligned_data.items():
    print(f"\nDataset: {animalID} {date}")
    print(f"Aligned imaging shape: {data['aligned_imaging'].shape}")
    print(f"Number of good trials: {len(data['good_trials'])}")


In [ ]:
import numpy as np

# Prepare RNN-ready inputs: (trials, time, neurons)
rnn_inputs = {}
for (animalID, date), data in aligned_data.items():
    aligned_imaging = data['aligned_imaging']  # shape: trials x neurons x frames
    rnn_inputs[(animalID, date)] = np.transpose(aligned_imaging, (0, 2, 1))
    print(
        f"RNN input for {animalID} {date}: {rnn_inputs[(animalID, date)].shape}"
    )
